# 관광 데이터 ETL 최소 버전 실습

 `load_data() → transform() → save_to_parquet()`


In [14]:
import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("준비 완료")

준비 완료


## 1. `isinstance()` 결과 확인

JSON에는 딕셔너리와 리스트가 들어갈 수 있습니다. 값을 직접 넣고 자료형 판별 결과를 확인합니다.

In [15]:
sample_list = ["해변", "가족여행"]
sample_dict = {"city": "속초", "lat": 38.2}

print(isinstance(sample_list, list))
print(isinstance(sample_dict, dict))

True
True


두 결과가 모두 `True`이면 리스트와 딕셔너리를 정상적으로 판별한 것입니다.

## 2. 중첩 관광 JSON 샘플 만들기
### ★ sample_data ★

In [ ]:
sample_data = [
    {
        "name": "속초해변",
        "location": {
            "city": "속초",
            "lat": 38.2,
        },
        "tags": ["해변", "가족여행"],
    }
]

sample_path = DATA_DIR / "sample_tour.json" # 경로 저장

with open(sample_path, "w", encoding="utf-8") as file:
    json.dump(sample_data, file, ensure_ascii=False, indent=2)

print("샘플 저장 경로:", sample_path)

샘플 저장 경로: data\sample_tour.json


## 3. 평탄화와 리스트 행 분리를 먼저 실험하기

In [17]:
raw_df = pd.DataFrame(sample_data)
display(raw_df)

print("location 자료형:", type(raw_df.loc[0, "location"]))
print("tags 자료형:", type(raw_df.loc[0, "tags"]))

,name,location,tags
0,속초해변,"{'city': '속초', 'lat': 38.2}","[해변, 가족여행]"


location 자료형: <class 'dict'>
tags 자료형: <class 'list'>


In [ ]:
# 중첩된 JSON 또는 딕셔너리 구조를 평평한 표 구조로 변환
flat_df = pd.json_normalize(
    raw_df.to_dict(orient="records"),
    sep=".",
)

display(flat_df)
print(flat_df.columns.tolist()) # 파이썬 리스트로 변환

,name,tags,location.city,location.lat
0,속초해변,"[해변, 가족여행]",속초,38.2


['name', 'tags', 'location.city', 'location.lat']


In [ ]:
# flat_df에서 리스트가 들어있는 컬럼을 찾아서 tag 리스트의 원소를 각각의 행으로 펼침
list_columns = [
    column
    for column in flat_df.columns
    if flat_df[column].apply(
        lambda value: isinstance(value, list)
    ).any()
]

print("리스트 컬럼:", list_columns)

preview_df = flat_df.explode("tags", ignore_index=True)
display(preview_df)

리스트 컬럼: ['tags']


,name,tags,location.city,location.lat
0,속초해변,해변,속초,38.2
1,속초해변,가족여행,속초,38.2


---

## 4. 최소 `TourDataETL` 클래스

읽기, 변환, 저장 세 메서드만 정의합니다. `_read_csv()` 같은 별도 읽기 함수와 특정 데이터셋 전용 함수는 만들지 않습니다.

In [ ]:
class TourDataETL:
    # 파일 읽기
    def load_data(self, file_path):
        """파일 확장자에 맞춰 데이터를 DataFrame으로 불러온다."""
        file_path = Path(file_path)
        suffix = file_path.suffix.lower() # 파일 경로에서 마지막 확장자를 가져와 소문자로 변환한 뒤 suffix 변수에 저장

        if suffix == ".csv": # 파일이 csv
            try:
                return pd.read_csv(file_path, encoding="utf-8-sig") #dataframe으로 반환
            except UnicodeDecodeError: #utf-8-sig로 읽을 수 없는 경우 cp949로 시도
                return pd.read_csv(file_path, encoding="cp949")

        if suffix == ".json": # 파일이 json
            with open(file_path, "r", encoding="utf-8") as file:
                data = json.load(file)

            if isinstance(data, list): # data가 list이면 dataframe으로 변환
                return pd.DataFrame(data)

            if isinstance(data, dict): # data가 dict이면 dataframe으로 변환
                return pd.DataFrame([data])

        if suffix == ".xml": # 파일이 xml 
            return pd.read_xml(file_path)

        if suffix == ".parquet": # 파일이 parquet
            return pd.read_parquet(file_path)

        raise ValueError(f"지원하지 않는 파일 형식입니다: {suffix}")

    # 데이터 정리/변환
    def transform(self, df):
        """빈 행·열을 제거하고 중첩 데이터를 표 형태로 변환한다."""
        if not isinstance(df, pd.DataFrame):
            raise TypeError("pandas DataFrame을 입력해야 합니다.")

        transformed_df = df.copy() # 복사본
        transformed_df = transformed_df.dropna(how="all") #전체가 비어있는 행 제거
        transformed_df = transformed_df.dropna(axis=1, how="all") #전체가 비어있는 열 제거

        # 중첩 딕셔너리를 독립된 컬럼으로 분리한다.
        transformed_df = pd.json_normalize(
            transformed_df.to_dict(orient="records"),
            sep=".",
        )

        # 최소 버전에서는 리스트 값을 각각의 행으로 분리한다. (위에 실험했던 코드와 동일)
        list_columns = [
            column
            for column in transformed_df.columns
            if transformed_df[column].apply(
                lambda value: isinstance(value, list)
            ).any()
        ]

        for column in list_columns:
            transformed_df = transformed_df.explode(
                column,
                ignore_index=True,
            )

        # 리스트 내부에 딕셔너리가 있던 경우 다시 컬럼으로 분리한다.
        transformed_df = pd.json_normalize(
            transformed_df.to_dict(orient="records"),
            sep=".",
        )

        return transformed_df.reset_index(drop=True)

    # 결과 저장
    def save_to_parquet(self, df, out_data_path):
        """DataFrame을 Parquet 파일로 저장한다."""
        out_data_path = Path(out_data_path)
        out_data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(out_data_path, index=False)
        return out_data_path

## 5. 샘플 파이프라인 실행

In [21]:
etl = TourDataETL()

loaded_df = etl.load_data(sample_path)
clean_df = etl.transform(loaded_df)
saved_path = etl.save_to_parquet(
    clean_df,
    OUTPUT_DIR / "sample_tour.parquet",
)

print("로드 결과:", loaded_df.shape)
print("변환 결과:", clean_df.shape)
print("저장 경로:", saved_path)
display(clean_df)

로드 결과: (1, 3)
변환 결과: (2, 4)
저장 경로: outputs\sample_tour.parquet


,name,tags,location.city,location.lat
0,속초해변,해변,속초,38.2
1,속초해변,가족여행,속초,38.2


In [22]:
check_df = pd.read_parquet(saved_path)

assert clean_df.equals(check_df)
assert clean_df.shape == (2, 4)
assert clean_df["tags"].tolist() == ["해변", "가족여행"]

print("최소 ETL 파이프라인 검증 통과")

최소 ETL 파이프라인 검증 통과


## 코드 검증

이 노트북의 모든 셀이 정상 실행된 다음 실제 관광 API의 JSON 응답 구조를 확인합니다. API 호출 코드를 먼저 클래스에 넣지 않고, 노트북에서 응답을 출력한 뒤 현재 `transform()`으로 처리할 수 있는지 판단합니다.

[개발계정]한국관광공사_국문 관광정보 서비스_GW (지역기반 관광정보 조회 데이터만 사용)
https://www.data.go.kr/iim/api/selectAPIAcountView.do

In [ ]:
import requests

# 공공데이터포털의 일반 인증키(Decoding)를 입력
SERVICE_KEY = "e717dac3c1ba8a11e4a6831d0f17bde48df21c742215e1dec683f7a79d868dff"

url = (
    "https://apis.data.go.kr/"
    "B551011/KorService2/areaBasedList2"
)

#요청 조건(params) 설정
params = {
    "serviceKey": SERVICE_KEY,
    "MobileOS": "ETC", # 운영체제 정보
    "MobileApp": "TourDataETL", #API를 호출하는 서비스명 = 지금 코드
    "_type": "json",
    "areaCode": "39",       # 제주특별자치도 지역코드
    "contentTypeId": "12",  # 관광지 (콘텐츠 종류)
    "numOfRows": 5,     # 한 번에 가져올 데이터 수
    "pageNo": 1,    # 페이지 번호
    "arrange": "A",         # 제목순
}

#서버 요청
response = requests.get(
    url,
    params=params,
    timeout=30, # 최대 30초까지 기다림
)

print("HTTP 상태 코드:", response.status_code)
print("응답 형식:", response.headers.get("Content-Type"))
print("응답 내용 일부:")
print(response.text[:1000])

HTTP 상태 코드: 200
응답 형식: application/json
응답 내용 일부:
{"response": {"header":{"resultCode":"0000","resultMsg":"OK"},"body": {"items": {"item":[{"addr1":"제주특별자치도 제주시 한경면 청수리","addr2":"1205","areacode":"39","cat1":"A01","cat2":"A0101","cat3":"A01010400","contentid":"1884191","contenttypeid":"12","createdtime":"20140113184533","firstimage":"http://tong.visitkorea.or.kr/cms/resource/72/3477972_image2_1.jpg","firstimage2":"http://tong.visitkorea.or.kr/cms/resource/72/3477972_image3_1.jpg","cpyrhtDivCd":"Type3","mapx":"126.2466098987","mapy":"33.3057279944","mlevel":"6","modifiedtime":"20250430134222","sigungucode":"4","tel":"","title":"가마오름","zipcode":"63007","lDongRegnCd":"50","lDongSignguCd":"110","lclsSystm1":"NA","lclsSystm2":"NA01","lclsSystm3":"NA010100"},{"addr1":"제주특별자치도 서귀포시 표선면 가시리","addr2":"","areacode":"39","cat1":"A01","cat2":"A0101","cat3":"A01010400","contentid":"1884505","contenttypeid":"12","createdtime":"20140114192329","firstimage":"http://tong.visitkorea.or.kr/cms/resource/9

In [24]:
# JSON 계층 구조와 자료형 확인
api_json = response.json()

print("전체 응답 자료형:", type(api_json))
print("최상위 키:", api_json.keys())

response_data = api_json["response"]
print("\nresponse 자료형:", type(response_data))
print("response의 키:", response_data.keys())

body_data = response_data["body"]
print("\nbody 자료형:", type(body_data))
print("body의 키:", body_data.keys())

items_data = body_data["items"]["item"]
print("\nitem 자료형:", type(items_data))
print("관광지 개수:", len(items_data))

print("\n첫 번째 관광지 자료형:", type(items_data[0]))
print("첫 번째 관광지의 키:")
print(items_data[0].keys())

print("\n첫 번째 관광지:")
display(items_data[0])

전체 응답 자료형: <class 'dict'>
최상위 키: dict_keys(['response'])

response 자료형: <class 'dict'>
response의 키: dict_keys(['header', 'body'])

body 자료형: <class 'dict'>
body의 키: dict_keys(['items', 'numOfRows', 'pageNo', 'totalCount'])

item 자료형: <class 'list'>
관광지 개수: 5

첫 번째 관광지 자료형: <class 'dict'>
첫 번째 관광지의 키:
dict_keys(['addr1', 'addr2', 'areacode', 'cat1', 'cat2', 'cat3', 'contentid', 'contenttypeid', 'createdtime', 'firstimage', 'firstimage2', 'cpyrhtDivCd', 'mapx', 'mapy', 'mlevel', 'modifiedtime', 'sigungucode', 'tel', 'title', 'zipcode', 'lDongRegnCd', 'lDongSignguCd', 'lclsSystm1', 'lclsSystm2', 'lclsSystm3'])

첫 번째 관광지:


{'addr1': '제주특별자치도 제주시 한경면 청수리',
 'addr2': '1205',
 'areacode': '39',
 'cat1': 'A01',
 'cat2': 'A0101',
 'cat3': 'A01010400',
 'contentid': '1884191',
 'contenttypeid': '12',
 'createdtime': '20140113184533',
 'firstimage': 'http://tong.visitkorea.or.kr/cms/resource/72/3477972_image2_1.jpg',
 'firstimage2': 'http://tong.visitkorea.or.kr/cms/resource/72/3477972_image3_1.jpg',
 'cpyrhtDivCd': 'Type3',
 'mapx': '126.2466098987',
 'mapy': '33.3057279944',
 'mlevel': '6',
 'modifiedtime': '20250430134222',
 'sigungucode': '4',
 'tel': '',
 'title': '가마오름',
 'zipcode': '63007',
 'lDongRegnCd': '50',
 'lDongSignguCd': '110',
 'lclsSystm1': 'NA',
 'lclsSystm2': 'NA01',
 'lclsSystm3': 'NA010100'}

In [ ]:
#전체 API 응답을 JSON 파일로 저장
api_json_path = Path("data") / "tour_api_sample.json" # JSON 저장 경로 만들기

with open(
    api_json_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump( # json.dump()는 Python의 dict/list 데이터를 JSON 파일에 저장하는 함수
        api_json,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("JSON 저장 경로:", api_json_path)


# 최소 ETL 파이프라인 실행
etl = TourDataETL()

api_raw = etl.load_data(api_json_path)

print("\n로드 결과 크기:", api_raw.shape)
print("로드 결과:")
display(api_raw)


api_transformed = etl.transform(api_raw)

print("\n변환 결과 크기:", api_transformed.shape)
print("변환된 컬럼:")
print(api_transformed.columns.tolist())

display(api_transformed.head())

JSON 저장 경로: data\tour_api_sample.json

로드 결과 크기: (1, 1)
로드 결과:


,response
0,"{'header': {'resultCode': '0000', 'resultMsg':..."



변환 결과 크기: (5, 30)
변환된 컬럼:
['response.header.resultCode', 'response.header.resultMsg', 'response.body.numOfRows', 'response.body.pageNo', 'response.body.totalCount', 'response.body.items.item.addr1', 'response.body.items.item.addr2', 'response.body.items.item.areacode', 'response.body.items.item.cat1', 'response.body.items.item.cat2', 'response.body.items.item.cat3', 'response.body.items.item.contentid', 'response.body.items.item.contenttypeid', 'response.body.items.item.createdtime', 'response.body.items.item.firstimage', 'response.body.items.item.firstimage2', 'response.body.items.item.cpyrhtDivCd', 'response.body.items.item.mapx', 'response.body.items.item.mapy', 'response.body.items.item.mlevel', 'response.body.items.item.modifiedtime', 'response.body.items.item.sigungucode', 'response.body.items.item.tel', 'response.body.items.item.title', 'response.body.items.item.zipcode', 'response.body.items.item.lDongRegnCd', 'response.body.items.item.lDongSignguCd', 'response.body.items.item

,response.header.resultCode,response.header.resultMsg,response.body.numOfRows,response.body.pageNo,response.body.totalCount,response.body.items.item.addr1,response.body.items.item.addr2,response.body.items.item.areacode,response.body.items.item.cat1,response.body.items.item.cat2,...,response.body.items.item.modifiedtime,response.body.items.item.sigungucode,response.body.items.item.tel,response.body.items.item.title,response.body.items.item.zipcode,response.body.items.item.lDongRegnCd,response.body.items.item.lDongSignguCd,response.body.items.item.lclsSystm1,response.body.items.item.lclsSystm2,response.body.items.item.lclsSystm3
0,0000,OK,5,1,287,제주특별자치도 제주시 한경면 청수리,1205,39,A01,A0101,...,20250430134222,4,,가마오름,63007,50,110,NA,NA01,NA010100
1,0000,OK,5,1,287,제주특별자치도 서귀포시 표선면 가시리,,39,A01,A0101,...,20250522111143,3,,가문이오름(감은이오름),63623,50,130,NA,NA01,NA010100
2,0000,OK,5,1,287,제주특별자치도 제주시 오라삼동,2848,39,A01,A0101,...,20251230161028,4,,가새기오름,63148,50,110,NA,NA01,NA010100
3,0000,OK,5,1,287,제주특별자치도 서귀포시 표선면 녹산로 554,,39,A01,A0101,...,20250520165953,3,,가세오름,63622,50,130,NA,NA01,NA010100
4,0000,OK,5,1,287,제주특별자치도 서귀포시 표선면 녹산로 464-78,,39,A02,A0204,...,20250408135047,3,,가시리국산화풍력발전단지,63622,50,130,EX,EX06,EX060900


In [26]:
# parquet 파일로 저장과 재로드 검증
api_output_path = etl.save_to_parquet(
    api_transformed,
    "outputs/tour_api_sample.parquet",
)

print("Parquet 저장 경로:", api_output_path)


# 저장한 Parquet 파일 다시 불러오기
api_check = pd.read_parquet(api_output_path)

print("재로드 결과 크기:", api_check.shape)

display(
    api_check[
        [
            "response.body.items.item.title",
            "response.body.items.item.addr1",
            "response.body.items.item.mapx",
            "response.body.items.item.mapy",
        ]
    ]
)


# 저장 전후 결과 검증
assert api_check.shape == api_transformed.shape

assert (
    api_check.columns.tolist()
    == api_transformed.columns.tolist()
)

assert len(api_check) == 5

assert (
    api_check["response.body.items.item.title"]
    .notna()
    .all()
)

# dict 또는 list가 셀 안에 남아 있는지 확인
nested_value_exists = api_check.map(
    lambda value: isinstance(
        value,
        (dict, list),
    )
).any().any()

assert not bool(nested_value_exists)

print("실제 관광 API ETL 파이프라인 검증 통과")


Parquet 저장 경로: outputs\tour_api_sample.parquet
재로드 결과 크기: (5, 30)


,response.body.items.item.title,response.body.items.item.addr1,response.body.items.item.mapx,response.body.items.item.mapy
0,가마오름,제주특별자치도 제주시 한경면 청수리,126.2466098987,33.3057279944
1,가문이오름(감은이오름),제주특별자치도 서귀포시 표선면 가시리,126.6876111209,33.4018299117
2,가새기오름,제주특별자치도 제주시 오라삼동,126.5038611983,33.4893700755
3,가세오름,제주특별자치도 서귀포시 표선면 녹산로 554,126.7231832338,33.3931758821
4,가시리국산화풍력발전단지,제주특별자치도 서귀포시 표선면 녹산로 464-78,126.7341100468,33.3939370848


실제 관광 API ETL 파이프라인 검증 통과
